# SDF Utilities

`isoext.sdf` provides **optional** utilities for working with signed distance functions. These are convenient for testing and demos, but you can always compute values with your own code.

**Note:** These utilities are not required — see [Working with Grids](grids.ipynb) for examples using raw PyTorch.


In [1]:
import isoext
from isoext.sdf import *
from isoext import viewer

grid = isoext.UniformGrid([128, 128, 128])


## Primitives

### SphereSDF
```python
SphereSDF(radius: float)
```


In [2]:
sphere = SphereSDF(radius=0.7)
grid.set_values(sphere(grid.get_points()))
v, f = isoext.marching_cubes(grid)
viewer.embed(v, f)


### TorusSDF
```python
TorusSDF(R: float, r: float)  # R=major radius, r=tube radius
```


In [3]:
torus = TorusSDF(R=0.6, r=0.2)
grid.set_values(torus(grid.get_points()))
v, f = isoext.marching_cubes(grid)
viewer.embed(v, f, color="gold")


### CuboidSDF
```python
CuboidSDF(size: list[float])  # Full size in [x, y, z]
```


In [4]:
cube = CuboidSDF(size=[1.0, 1.0, 1.0])
grid.set_values(cube(grid.get_points()))
v, f = isoext.marching_cubes(grid)
viewer.embed(v, f, color="salmon")


### MandelbulbSDF

A distance estimator for the Mandelbulb fractal. The values approximate
the distance to the surface rather than being an exact SDF; fewer
iterations give a smoother shape.


In [5]:
bulb = MandelbulbSDF(iterations=6)

# The bulb needs slightly larger bounds than the shared grid above
bulb_grid = isoext.UniformGrid([128, 128, 128], aabb_min=[-1.2, -1.2, -1.2], aabb_max=[1.2, 1.2, 1.2])
bulb_grid.set_values(bulb(bulb_grid.get_points()))
v, f = isoext.marching_cubes(bulb_grid)
viewer.embed(v, f, color="coral")


## CSG Operations

Combine shapes using Constructive Solid Geometry:

| Operation | Description |
|-----------|-------------|
| `UnionOp([...])` | Combine shapes (min of SDFs) |
| `IntersectionOp([...])` | Keep overlap (max of SDFs) |
| `NegationOp(sdf)` | Invert inside/outside |
| `SmoothUnionOp([...], k)` | Smooth blend with radius k |


In [6]:
# Sphere with a hole drilled through it
sphere = SphereSDF(radius=0.7)
hole = CuboidSDF(size=[0.3, 0.3, 2.0])
drilled = IntersectionOp([sphere, NegationOp(hole)])

grid.set_values(drilled(grid.get_points()))
v, f = isoext.marching_cubes(grid)
viewer.embed(v, f, color="orchid")


## Transformations

| Transform | Description |
|-----------|-------------|
| `TranslationOp(sdf, offset)` | Move by `[x, y, z]` |
| `RotationOp(sdf, axis, angle)` | Rotate around axis (degrees by default) |


In [7]:
# Two spheres with smooth blending
s1 = TranslationOp(SphereSDF(radius=0.4), offset=[-0.3, 0, 0])
s2 = TranslationOp(SphereSDF(radius=0.4), offset=[0.3, 0, 0])
blended = SmoothUnionOp([s1, s2], k=0.15)

grid.set_values(blended(grid.get_points()))
v, f = isoext.marching_cubes(grid)
viewer.embed(v, f, color="tomato")
